#### Simple Gen AI APP Using Langchain

In [2]:
import os
from dotenv import load_dotenv
load_dotenv()

## Langsmith Tracking
os.environ["LANGCHAIN_API_KEY"]=os.getenv("LANGCHAIN_API_KEY")
os.environ["LANGCHAIN_TRACING_V2"]="true"
os.environ["LANGCHAIN_PROJECT"]=os.getenv("LANGCHAIN_PROJECT")

In [3]:
## Data Ingestion--From the website we need to scrape the data
from langchain_community.document_loaders import WebBaseLoader

USER_AGENT environment variable not set, consider setting it to identify your requests.


In [4]:
loader=WebBaseLoader("https://docs.smith.langchain.com/pricing/faq")
loader

In [5]:
docs=loader.load()
docs

[Document(metadata={'source': 'https://docs.smith.langchain.com/pricing/faq', 'title': 'Plans and Pricing - LangChain', 'description': "Pricing for LangChain products for teams of any size. Choose the plan that suits your needs, whether you're an individual developer or enterprise.", 'language': 'en'}, page_content='Plans and Pricing - LangChain\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\nProducts\n\nOpen Source FrameworksLangChainQuick start agents with any model providerLangGraphBuild custom agents with low-level controlDeep AgentsNewUse planning, memory, and sub-agents for complex, long-running tasksLangSmithObservabilityDebug and monitor in-depth tracesEvaluationIterate on prompts and modelsDeploymentShip and scale agents in productionResources\n\nLangChain AcademyBlogCustomer StoriesCommunityEventsChangelogGuidesTrust CenterDocsCompany\n\nAboutCareersPricingTalk to salesSign upPlans for teams of\xa0any\xa0sizeGet all LangSmith services - pay for what you useDeveloperFor sol

In [6]:
### Load Data--> Docs-->Divide our Docuemnts into chunks dcouments-->text-->vectors-->Vector Embeddings--->Vector Store DB
from langchain_text_splitters import RecursiveCharacterTextSplitter

text_splitter=RecursiveCharacterTextSplitter(chunk_size=500,chunk_overlap=50)
documents = text_splitter.split_documents(docs)

In [8]:
type(documents[0])

langchain_core.documents.base.Document

In [7]:
from langchain_ollama import OllamaEmbeddings
embeddings = OllamaEmbeddings(model = 'mxbai-embed-large')

In [8]:
from langchain_community.vectorstores import FAISS
vectorstoredb=FAISS.from_documents(documents,embeddings)

In [9]:
vectorstoredb

In [10]:
## Query From a vector db
query="ingested event"
result=vectorstoredb.similarity_search(query)
result[0].page_content

'ObservabilityLangSmith EvaluationLangSmith DeploymentResourcesGuidesBlogCustomer StoriesLangChain AcademyCommunityEventsChangelogDocsCompanyAboutCareersXLinkedInYouTubeMarketing AssetsSecuritySign up for our newsletter to stay up to dateThank you! Your submission has been received!Oops! Something went wrong while submitting the form.All systems operationalPrivacy PolicyTerms of Service'

In [11]:
print(len(result))

4


In [14]:
from langchain_ollama import OllamaLLM
llm = OllamaLLM(model = 'gemma3:1b')
llm.invoke('Hi, What is my name?')

"I understand you're asking this question, but I'm programmed to protect your privacy and cannot share personal information like your name. It’s important for me to maintain confidentiality. \n\nI can, however, offer you some general information about me! I’m a large language model created by Google. \n\nDo you want to talk about something else? Perhaps you’d like to:\n\n*   Ask me a question?\n*   Tell me about a topic you enjoy?"

In [15]:
## Retrieval Chain, Document chain

from langchain.chains.combine_documents import create_stuff_documents_chain
from langchain_core.prompts import ChatPromptTemplate

prompt = ChatPromptTemplate.from_template(
    """
Answer the following question based only on the provided context:
# <context
{context}
</context>


"""
)

document_chain=create_stuff_documents_chain(llm,prompt)
document_chain

RunnableBinding(bound=RunnableBinding(bound=RunnableAssign(mapper={
  context: RunnableLambda(format_docs)
}), kwargs={}, config={'run_name': 'format_inputs'}, config_factories=[])
| ChatPromptTemplate(input_variables=['context'], input_types={}, partial_variables={}, messages=[HumanMessagePromptTemplate(prompt=PromptTemplate(input_variables=['context'], input_types={}, partial_variables={}, template='\nAnswer the following question based only on the provided context:\n# <context\n{context}\n</context>\n\n\n'), additional_kwargs={})])
| OllamaLLM(model='gemma3:1b')
| StrOutputParser(), kwargs={}, config={'run_name': 'stuff_documents_chain'}, config_factories=[])

In [16]:
from langchain_core.documents import Document
document_chain.invoke({
    "input":"I’ve hit my rate or usage limits. What can I do?",
    "context":[Document(page_content="When you first sign up for a LangSmith account, you get a Personal organization that is limited to 5000 monthly traces. To continue sending traces after reaching this limit, upgrade to the Developer or Plus plans by adding a credit card. Head to Plans and Billing to upgrade.")]
})

'To continue sending traces after reaching 5000 monthly traces, you need to upgrade to the Developer or Plus plans.'

However, we want the documents to first come from the retriever we just set up. That way, we can use the retriever to dynamically select the most relevant documents and pass those in for a given question.

In [32]:
### Input--->Retriever--->vectorstoredb

vectorstoredb

In [17]:
retriever=vectorstoredb.as_retriever()
from langchain.chains import create_retrieval_chain
retrieval_chain=create_retrieval_chain(retriever,document_chain)


In [18]:
retrieval_chain

RunnableBinding(bound=RunnableAssign(mapper={
  context: RunnableBinding(bound=RunnableLambda(lambda x: x['input'])
           | VectorStoreRetriever(tags=['FAISS', 'OllamaEmbeddings'], vectorstore=<langchain_community.vectorstores.faiss.FAISS object at 0x000002BD7F3412D0>, search_kwargs={}), kwargs={}, config={'run_name': 'retrieve_documents'}, config_factories=[])
})
| RunnableAssign(mapper={
    answer: RunnableBinding(bound=RunnableBinding(bound=RunnableAssign(mapper={
              context: RunnableLambda(format_docs)
            }), kwargs={}, config={'run_name': 'format_inputs'}, config_factories=[])
            | ChatPromptTemplate(input_variables=['context'], input_types={}, partial_variables={}, messages=[HumanMessagePromptTemplate(prompt=PromptTemplate(input_variables=['context'], input_types={}, partial_variables={}, template='\nAnswer the following question based only on the provided context:\n# <context\n{context}\n</context>\n\n\n'), additional_kwargs={})])
            |

In [19]:
## Get the response form the LLM
response=retrieval_chain.invoke({"input":"I’ve hit my rate or usage limits. What can I do?"})
response

{'input': 'I’ve hit my rate or usage limits. What can I do?',
 'context': [Document(id='a9244793-9bbe-497f-a959-ebdc7bc79327', metadata={'source': 'https://docs.smith.langchain.com/pricing/faq', 'title': 'Plans and Pricing - LangChain', 'description': "Pricing for LangChain products for teams of any size. Choose the plan that suits your needs, whether you're an individual developer or enterprise.", 'language': 'en'}, page_content='the right retention for each trace, helping you balance cost and value. I’ve exhausted my free trace allocation. What can I do?'),
  Document(id='1a509e33-4b83-4704-a1ec-acd54bf4477e', metadata={'source': 'https://docs.smith.langchain.com/pricing/faq', 'title': 'Plans and Pricing - LangChain', 'description': "Pricing for LangChain products for teams of any size. Choose the plan that suits your needs, whether you're an individual developer or enterprise.", 'language': 'en'}, page_content='If you’ve used up your free traces, you can input your credit card detai

In [20]:
print(response['answer'])

You might want to upgrade a base trace to an extended trace if you want to balance cost and value.


In [21]:
response['context']

[Document(id='a9244793-9bbe-497f-a959-ebdc7bc79327', metadata={'source': 'https://docs.smith.langchain.com/pricing/faq', 'title': 'Plans and Pricing - LangChain', 'description': "Pricing for LangChain products for teams of any size. Choose the plan that suits your needs, whether you're an individual developer or enterprise.", 'language': 'en'}, page_content='the right retention for each trace, helping you balance cost and value. I’ve exhausted my free trace allocation. What can I do?'),
 Document(id='1a509e33-4b83-4704-a1ec-acd54bf4477e', metadata={'source': 'https://docs.smith.langchain.com/pricing/faq', 'title': 'Plans and Pricing - LangChain', 'description': "Pricing for LangChain products for teams of any size. Choose the plan that suits your needs, whether you're an individual developer or enterprise.", 'language': 'en'}, page_content='If you’ve used up your free traces, you can input your credit card details on the Developer or Plus plans to continue sending traces to LangSmith. 